# Deep-Agents × Forge — hands-on notebook

Runs a real [`deepagents`](https://github.com/langchain-ai/deepagents) agent whose sandbox is backed by a Forge workspace. The agent writes Python files inside an isolated workspace, executes them via `docker exec` on a pooled container, and reads the output back.

**Model provider:** the local OpenAI-compatible endpoint at `http://localhost:6655/openai/v1` — no external calls.

**What this proves end-to-end:**
- `ForgeSandbox` subclasses `deepagents.backends.sandbox.BaseSandbox`, so `create_deep_agent(backend=...)` just works.
- Every tool call (`write`, `execute`, `read`, `ls`, `grep`, ...) resolves inside a Forge workspace.
- The container pool is shared — spin up a second agent in the same notebook and both share the warm containers.

## Prerequisites

```bash
uv sync --all-extras --dev
docker pull python:3.14-slim   # or set FORGE_TEST_IMAGE
```

Then choose ONE of the two ways to run Forge:

- **Embedded** (simplest — no daemon; this notebook wires everything in-process). Just run the cells.
- **Daemon** — in a separate terminal: `uv run forged serve --pool-max 4`. Then replace `Forge.local()` below with `Forge("http://127.0.0.1:8787")`.

## 1. Configure the model provider

Point `langchain-openai` at the local endpoint. The `OPENAI_*` env vars are read by `ChatOpenAI` internally.

In [ ]:
import os

os.environ["OPENAI_BASE_URL"] = "http://localhost:6655/openai/v1"
os.environ["OPENAI_API_KEY"] = "5e7a770f-3b44-480b-a07f-0653340321ac"

MODEL_NAME = "gpt-5"                 # or gpt-5-mini / gpt-4.1 / gpt-4.1-mini / gpt-5.5 / gpt-5.4
IMAGE = "python:3.14-slim"           # container image the agent runs commands in

print(f"model = {MODEL_NAME}\nimage = {IMAGE}")

## 2. Bring up Forge (embedded) and build a Deep-Agents sandbox

`Forge.local(...)` runs the metastore, container pool, and Docker driver inside this Python process — no HTTP hop. If you'd rather use the daemon, swap this cell for `forge = Forge("http://127.0.0.1:8787")` and `await forge.__aenter__()`.

`ForgeSandbox.afrom_thread(...)` creates a fresh workspace tagged with the thread id — which is exactly how a LangGraph agent would model one conversation.

In [ ]:
import tempfile
from pathlib import Path

from forge.client import Forge
from forge.config import make_config
from forge.langchain import ForgeSandbox
from forge.models import PoolConfig

# Fresh scratch directory for the metastore + workspaces
_TMP = tempfile.mkdtemp(prefix="forge-notebook-")
print(f"forge data root: {_TMP}")

cfg = make_config(
    Path(_TMP),
    default_pool=PoolConfig(
        image=IMAGE,
        min_idle=1,
        max_size=4,             # 4 warm containers, shared across everyone
        idle_ttl_seconds=600,
        exec_timeout_seconds=60,
        max_output_bytes=100_000,
    ),
)

forge = Forge.local(config=cfg)
await forge.__aenter__()   # boot the pool + services in the notebook's loop

sandbox = await ForgeSandbox.afrom_thread(
    forge=forge, thread_id="notebook-thread-1", image=IMAGE
)
print(f"workspace id: {sandbox.workspace.id}")

## 3. Sanity check — no LLM yet

Prove the sandbox is wired: write a file, `python`-execute it, read it back. If this cell fails, don't bother running the agent — something's off with Docker or the pool.

In [ ]:
wr = await sandbox.awrite("hello.py", "print('hi from forge')\n")
assert wr.error is None, wr.error

result = await sandbox.aexecute("python hello.py")
print("exit_code:", result.exit_code)
print("output:   ", result.output.strip())

read = await sandbox.aread("hello.py")
print("file content:", (read.file_data or {}).get("content", "").strip())

## 4. Build the Deep-Agents agent

This is the money cell. `create_deep_agent(backend=sandbox)` wires the Forge sandbox as the filesystem + shell backend. The LLM will call `write`, `read`, `execute`, `ls`, `grep`, etc. — all routed through `ForgeSandbox` — automatically.

In [ ]:
from deepagents import create_deep_agent
from langchain_openai import ChatOpenAI

# Uses OPENAI_BASE_URL + OPENAI_API_KEY from cell 1.
llm = ChatOpenAI(model=MODEL_NAME, temperature=0)

agent = create_deep_agent(
    model=llm,
    backend=sandbox,
    system_prompt=(
        "You are a coding assistant with access to a real sandboxed Linux "
        "filesystem and shell.\n"
        "- Write files with the write tool, not by echoing shell text.\n"
        "- Run Python with `python <file>`; check exit codes.\n"
        "- Report the final answer clearly, and mention the exit code."
    ),
)
print("agent ready.")

## 5. Run a task

Ask the agent to write, run, and report. Behind the scenes it will:

1. Call `write("fib.py", ...)` → `ForgeSandbox.awrite` → workspace filesystem.
2. Call `execute("python fib.py")` → `ForgeSandbox.aexecute` → lease a pooled container, `docker exec forge-run python fib.py`, stream output back.
3. Call `read("fib.py")` if it wants to double-check.

The final message is the assistant's plain-English summary.

In [ ]:
task = (
    "Create a Python script called fib.py that prints the 20th Fibonacci number "
    "(iteratively, not recursively). Run it and tell me the number plus the exit code."
)

result = await agent.ainvoke({"messages": [{"role": "user", "content": task}]})

final_message = result["messages"][-1]
print("=== assistant reply ===\n")
print(final_message.content)

## 6. Inspect the workspace after the run

The workspace is a real host directory that survives after the agent finishes. You can list files, snapshot it, restore it into a new workspace, or export artifacts — all via the Forge SDK.

In [ ]:
listing = await sandbox.workspace.files.ls(".")
for entry in listing:
    kind = "d" if entry.is_dir else "-"
    print(f"{kind} {entry.size_bytes:>8}  {entry.path}")

# Read fib.py back (if the agent created it):
try:
    r = await sandbox.workspace.files.read("fib.py")
    print("\n=== fib.py ===\n" + r.content)
except Exception as e:
    print("(fib.py not present:", e, ")")

## 7. Follow-up turn (same workspace)

The Deep-Agents state carries between turns, and the workspace persists on disk. Ask the agent to modify what it wrote.

In [ ]:
followup = (
    "Now modify fib.py to also print the first 10 Fibonacci numbers, one per line, "
    "before the 20th. Run it again and show the exact stdout."
)

result2 = await agent.ainvoke({
    "messages": result["messages"] + [{"role": "user", "content": followup}]
})

print("=== assistant reply ===\n")
print(result2["messages"][-1].content)

## 8. Snapshot the workspace

Snapshots are `tar.zst` archives on disk under `<data_root>/snapshots/`. `restore()` gives you a brand-new workspace with the same files.

In [ ]:
snap = await sandbox.workspace.snapshots.create(name="after-fib")
print(f"snapshot = {snap.id} ({snap.size_bytes} bytes)")

# Restore into a fresh workspace:
restored_model = await forge._transport.restore_snapshot(  # noqa: SLF001
    snap.id, name="restored-from-snap", metadata={}
)
restored = await forge.workspaces.get(restored_model.id)
verify = await restored.exec(["python", "fib.py"])
print(f"restored workspace = {restored.id}")
print(f"re-run exit = {verify.exit_code}")
print(f"re-run output (first 200 chars):\n{verify.output[:200]}")

## 9. Pool stats

This is the reason Forge exists: many agents share a small container pool. Look at `total_leases` — that's how many times the agent (plus our sanity checks) leased a container. `total` is the current container count — note it's usually 1 (the min_idle warm one) even though we peaked at max_size=4 during the burst.

In [ ]:
for stats in await forge.pool_status():
    print(
        f"image={stats.image}\n"
        f"  idle={stats.idle}  in_use={stats.in_use}  total={stats.total}  "
        f"max={stats.max_size}  min_idle={stats.min_idle}\n"
        f"  total_leases={stats.total_leases}  wait_ms={stats.total_lease_wait_ms}  "
        f"health_kills={stats.total_health_kills}"
    )

## 10. Cleanup

Tears down the pool (destroys warm containers) and closes the metastore. Skip this if you plan to keep iterating.

In [ ]:
await forge.__aexit__(None, None, None)
print("forge shut down cleanly.")

## Troubleshooting

- **`asyncio` error "cannot be called from a running event loop"** — make sure the notebook kernel supports top-level `await` (Jupyter Lab / VS Code notebooks do by default; the classic Notebook may need `nest_asyncio`).
- **`docker: daemon not reachable`** — start Docker Desktop / colima / OrbStack.
- **First exec is slow (~1 s)** — that's the cold container start. Subsequent execs reuse warm containers and land in ~30–100 ms.
- **LLM returns nonsense JSON tool calls** — some `gpt-5-mini` etc. endpoints skimp on tool-calling; try `gpt-5` or `gpt-4.1`.
- **Container isolation caveat** — all workspaces share one bind mount at `/workspaces`. Fine for single-tenant. V2 Firecracker fixes this.